In [1]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

def predict_masked_token(text: str, top_k: int = 1):
    """
    マスクされたトークンに対して、上位 top_k の予測を返します。
    text 中に [MASK] を含めてください。
    """
    # トークナイザとモデルをロード
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertForMaskedLM.from_pretrained('bert-base-uncased')
    model.eval()

    # トークナイズ・ID変換
    inputs = tokenizer.encode_plus(text, return_tensors='pt')
    input_ids = inputs['input_ids']
    mask_token_index = torch.where(input_ids == tokenizer.mask_token_id)[1]

    # 予測
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # マスク位置のロジットを取得し、上位 top_k のトークンを選択
    mask_logits = logits[0, mask_token_index, :]
    top_k_weights, top_k_indices = torch.topk(mask_logits, top_k, dim=1)

    results = []
    for i in range(top_k):
        token_id = top_k_indices[0, i].item()
        token = tokenizer.convert_ids_to_tokens([token_id])[0]
        score = top_k_weights[0, i].item()
        results.append((token, score))
    return results

if __name__ == "__main__":
    sentence = "The movie was full of [MASK]."
    predictions = predict_masked_token(sentence, top_k=5)
    for token, score in predictions:
        print(f"{token}\t(score: {score:.4f})")

c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\sklearn\utils\_param_validation.py:11: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  from scipy.sparse import csr_matrix, issparse
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in 

fun	(score: 9.2890)
surprises	(score: 8.8099)
drama	(score: 8.4146)
stars	(score: 7.9189)
laughs	(score: 7.8503)
